In [ ]:
%matplotlib inline
import sys, os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, '/Users/yotameviatar/vs_code')
warnings.filterwarnings('ignore')

from Airflow import airflow_config as config
from Airflow import airflow_processor as processor

plt.rcParams.update({'figure.dpi': 110})
print('Setup complete')

In [ ]:
# ── What to show ──────────────────────────────────────────────────────────────
SUBJECT   = 'AB22'    # subject ID
SESSION   = None      # 'eve', 'mor', or None for both

# ── Signal display ────────────────────────────────────────────────────────────
SHOW_NK2  = True      # overlay NK2-cleaned signal on raw (blue/red colour)
RAW_ONLY  = False     # True = grey raw only, no NK2 overlay

# ── Layout ───────────────────────────────────────────────────────────────────
NCOLS     = 6
SAVE_FIGS = True      # also write PNGs to OUTPUT_DIR/raw_show/<subject>/

In [ ]:
# ── Load cache — handles old format (trials) and new format (sessions) ────────
cache_path = os.path.join(config.OUTPUT_DIR, '_cache', 'airflow_cache.pkl')
with open(cache_path, 'rb') as f:
    _cache = pickle.load(f)

if 'sessions' in _cache:
    print('Cache format: new (sessions)')
    trials_data, traces = processor.apply_analysis_params(_cache['sessions'], config)
else:
    print('Cache format: old (trials) — running _run_analysis on cached epochs')
    raw_trials = _cache['trials']
    traces     = _cache.get('traces', {})
    trials_data = {}
    for subj, sess_dict in sorted(raw_trials.items()):
        trials_data[subj] = {}
        for sess_key, trial_list in sess_dict.items():
            trials = [dict(t) for t in trial_list]
            for t in trials:
                t['baseline_mean'] = None; t['score'] = None
                t['rejected'] = None; t['rejection_reason'] = ''
                t['epoch_anal'] = None; t['times_anal'] = None
            processor._run_analysis(trials, config)
            trials_data[subj][sess_key] = trials

if SUBJECT not in trials_data:
    raise ValueError(f"Subject '{SUBJECT}' not found. Available: {sorted(trials_data.keys())}")

show_nk2  = SHOW_NK2 and not RAW_ONLY
sess_keys = ([SESSION] if SESSION else
             sorted(k for k in trials_data[SUBJECT] if trials_data[SUBJECT][k]))

for sk in sess_keys:
    trials = trials_data[SUBJECT].get(sk, [])
    n_rej  = sum(1 for t in trials if t.get('rejected'))
    print(f"{SUBJECT} {config.SESSION_MAP.get(sk,{}).get('label', sk)}: "
          f"{len(trials)} trials, {n_rej} rejected")

---
## View 1 — Individual Trials per Session
Each subplot = one trial.  
**Grey** = raw unfiltered airflow · **Colour** = NK2-cleaned (khodadad2018 bandpass 0.05–3 Hz)  
**▲/▽** = NK2-detected breath peaks/troughs · **Blue shading** = baseline window · **Red shading** = response window  
Title colour: black = pass · grey = rejected

In [ ]:
label_name = {1: 'Neg', 2: 'Neu'}
label_col  = {1: config.NEG_COLOR, 2: config.NEU_COLOR}

for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    n     = len(trials)
    nrows = max(1, (n + NCOLS - 1) // NCOLS)
    fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 3.0, nrows * 2.4),
                             squeeze=False)
    axes_flat = axes.flatten()

    for i, t in enumerate(trials):
        ax        = axes_flat[i]
        tw        = t['times_wide'].astype(float)
        ep_raw    = t['epoch_raw_wide'].astype(float)
        ep_clean  = t.get('epoch_clean_wide')
        ep_peaks  = t.get('ep_peaks',   np.array([], dtype=int))
        ep_troughs= t.get('ep_troughs', np.array([], dtype=int))
        rej       = t.get('rejected')
        lbl       = t.get('label', 0)
        bm        = t.get('baseline_mean')
        sc        = t.get('score')
        rr        = t.get('rejection_reason', '')

        if rej:
            nk2_col = '#95A5A6';  title_col = 'dimgray'
        else:
            nk2_col = label_col.get(lbl, '#95A5A6');  title_col = 'black'

        # Raw signal (always grey)
        ax.plot(tw, ep_raw, color='#CCCCCC', lw=0.7, zorder=2, label='raw')

        # NK2 overlay
        if show_nk2 and ep_clean is not None:
            ax.plot(tw, ep_clean.astype(float), color=nk2_col, lw=1.1, zorder=3, label='NK2')
            valid_pk = ep_peaks[(ep_peaks >= 0) & (ep_peaks < len(ep_clean))]
            valid_tr = ep_troughs[(ep_troughs >= 0) & (ep_troughs < len(ep_clean))]
            if len(valid_pk) > 0:
                ax.plot(tw[valid_pk],  ep_clean[valid_pk],  '^',
                        color=nk2_col, ms=3.5, zorder=4, alpha=0.8)
            if len(valid_tr) > 0:
                ax.plot(tw[valid_tr], ep_clean[valid_tr], 'v',
                        color=nk2_col, ms=3.5, zorder=4, alpha=0.8)

        # Window shading
        ax.axvspan(config.BASELINE_TMIN,  config.BASELINE_TMAX,  alpha=0.10, color='#3498DB', zorder=0)
        ax.axvspan(config.RESPONSE_TMIN,  config.RESPONSE_TMAX,  alpha=0.08, color='#E74C3C',  zorder=0)
        if bm is not None and not np.isnan(float(bm)):
            ax.axhline(float(bm), color='goldenrod', ls=':', lw=0.9, alpha=0.85)
        ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.45)

        ax.set_xlim(tw[0], tw[-1])
        ax.tick_params(labelsize=5.5)
        ax.spines[['top', 'right']].set_visible(False)

        if rej:
            title = f'T{i+1} REJ\n{rr[:22] or "?"}'
        else:
            sc_str = f'{sc:.2f}' if sc is not None and not np.isnan(float(sc)) else '—'
            title  = f'T{i+1} {label_name.get(lbl,"?")} sc={sc_str}'
        ax.set_title(title, fontsize=6.5, color=title_col, pad=2)

    for ax in axes_flat[n:]:
        ax.set_visible(False)

    nk2_note = '  |  colour=NK2 clean   grey=raw' if show_nk2 else '  |  raw only'
    fig.suptitle(f'{SUBJECT} — {sess_label} — Individual trials  (N={n}){nk2_note}',
                 fontsize=11, fontweight='bold')

    fig.legend(handles=[
        plt.Line2D([0],[0], color='#CCCCCC', lw=1.5, label='Raw airflow'),
        plt.Line2D([0],[0], color=config.NEG_COLOR, lw=1.5, label='NK2 – Negative'),
        plt.Line2D([0],[0], color=config.NEU_COLOR, lw=1.5, label='NK2 – Neutral'),
        plt.Line2D([0],[0], color='#95A5A6', lw=1.5, label='NK2 – Rejected'),
        mpatches.Patch(color='#3498DB', alpha=0.5, label='Baseline window'),
        mpatches.Patch(color='#E74C3C', alpha=0.4, label='Response window'),
        plt.Line2D([0],[0], color='goldenrod', ls=':', label='Baseline mean'),
    ], fontsize=7, loc='lower center', ncol=7, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'raw_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_trials.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()

---
## View 2 — Full Session Overview
NK2-filtered airflow signal across the entire session.  
**Green lines** = startle onsets where trial passed · **Red lines** = rejected trials

In [ ]:
for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    tr = traces.get(SUBJECT, {}).get(sess_key)
    if tr is None or tr.get('signal') is None:
        print(f'{sess_key}: no session trace — skipping')
        continue

    sig    = tr['signal'].astype(float)
    sfreq  = tr['sfreq']
    t_axis = np.arange(len(sig)) / sfreq

    fig, ax = plt.subplots(figsize=(20, 3.5))
    ax.plot(t_axis, sig, color='#2C3E50', lw=0.6, zorder=2, label='NK2 filtered')

    startle_sec = tr['startle_samps_sec']
    n = min(len(startle_sec), len(trials))
    for idx in range(n):
        t_star = float(startle_sec[idx])
        rej    = trials[idx].get('rejected')
        col    = '#27AE60' if not rej else '#E74C3C'
        ax.axvline(t_star, color=col, lw=0.9, alpha=0.75, zorder=4)

    ax.set_xlabel('Time in session (s)', fontsize=9)
    ax.set_ylabel('Airflow (a.u.)', fontsize=9)
    n_rej = sum(1 for t in trials[:n] if t.get('rejected'))
    ax.set_title(
        f'{SUBJECT} — {sess_label} — Full session (NK2 filtered)  '
        f'(green=pass, red=reject  {n_rej}/{n} rejected)',
        fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'raw_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_session.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()